# Notebook 02: Understanding DPF Physics

**Prerequisite:** Notebook 01 (your first DPF simulation)  
**New topics:** Five discharge phases, neutron yield, beam-target vs thermonuclear fusion, PF-1000 validation  
**Time to complete:** ~45 minutes

---

## Overview

In this notebook we load the **PF-1000** preset — the 1-megajoule device at IPPLM Warsaw, Poland, which is the world's best-characterized large DPF. We use it to illustrate each phase of the discharge in detail, compare simulation against published experimental data, and decompose the neutron yield into its two physical mechanisms.

Understanding why the PF-1000 is important: it has been fired more than 12,000 times. Its current waveforms, neutron yields, and X-ray images have been published extensively. It is the standard benchmark device for DPF simulation codes worldwide.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from dpf.validation.lee_model_comparison import (
    LeeModel,
    estimate_neutron_yield_from_lee_result,
)
from dpf.validation.experimental import DEVICES
from dpf.presets import get_preset

# The YieldTracker decomposes yield into beam-target + thermonuclear
from dpf.diagnostics.yield_tracker import YieldTracker, YieldResult

print("Imports OK.")

## Step 1: Load and inspect the PF-1000 preset

The PF-1000 stores 1.13 MJ at 27 kV and delivers ~1.8 MA to the pinch. Compare these numbers to the tutorial device (3 kJ, 176 kA) to appreciate the scale difference.

In [ ]:
preset = get_preset('pf1000')
cc = preset['circuit']
sp = preset['snowplow']

# Fill pressure from density (ideal gas, D2)
m_D2  = 6.687e-27   # D2 molecule [kg]
k_B   = 1.381e-23   # Boltzmann constant [J/K]
rho0  = preset['rho0']
p_Pa  = rho0 * k_B * 300 / m_D2
p_torr = p_Pa / 133.322

E_stored = 0.5 * cc['C'] * cc['V0']**2

print("=== PF-1000 (IPPLM Warsaw) ===")
print(f"  Stored energy   : {E_stored/1e3:.0f} kJ  ({E_stored/1e6:.2f} MJ)")
print(f"  Charge voltage  : {cc['V0']/1e3:.0f} kV")
print(f"  Capacitance     : {cc['C']*1e3:.1f} mF")
print(f"  L₀              : {cc['L0']*1e9:.0f} nH")
print(f"  R₀              : {cc['R0']*1e3:.1f} mΩ")
print(f"  Anode radius  a : {cc['anode_radius']*100:.1f} cm")
print(f"  Cathode radius b: {cc['cathode_radius']*100:.1f} cm")
print(f"  Anode length    : {sp['anode_length']*100:.0f} cm")
print(f"  Fill pressure   : {p_torr:.1f} Torr")
print(f"  fc (current fraction): {sp['current_fraction']}")
print(f"  fm (mass fraction)   : {sp['mass_fraction']}")
print()

# Experimental peak current from the database
exp = DEVICES['PF-1000']
print(f"Experimental I_peak (published): {exp.peak_current/1e6:.2f} MA")

## Step 2: Run the Lee model for PF-1000

In [ ]:
device_params = {
    'C':              cc['C'],
    'V0':             cc['V0'],
    'L0':             cc['L0'],
    'R0':             cc['R0'],
    'anode_radius':   cc['anode_radius'],
    'cathode_radius': cc['cathode_radius'],
    'anode_length':   sp['anode_length'],
    'fill_pressure_torr': p_torr,
}

lee = LeeModel(
    current_fraction=sp['current_fraction'],
    mass_fraction=sp['mass_fraction'],
    radial_mass_fraction=sp.get('radial_mass_fraction', sp['mass_fraction']),
    pinch_column_fraction=sp.get('pinch_column_fraction', 1.0),
)

result = lee.run(device_params=device_params)

print(f"Simulated  I_peak : {result.peak_current/1e6:.3f} MA")
print(f"Experimental I_peak: {exp.peak_current/1e6:.3f} MA")
print(f"Error             : {abs(result.peak_current - exp.peak_current)/exp.peak_current*100:.1f}%")
print(f"Simulated  pinch  : {result.pinch_time*1e6:.2f} µs")
print(f"Phases completed  : {result.phases_completed}")

## Step 3: Identify the five discharge phases

The Lee model captures phases 1–4. We add a post-pinch phase (5) that begins after the pinch reaches minimum radius.

| Phase | Name | What is happening |
|-------|------|-------------------|
| 1 | Breakdown / liftoff | Gas ionizes. Current sheet forms at insulator sleeve. |
| 2 | Axial rundown | J×B force drives sheet toward anode tip. Snowplow accumulates gas. |
| 3 | Radial inward shock | Sheet implodes radially. Reflected shock accelerates inner slug. |
| 4 | Pinch | Plasma column at minimum radius. Fusion reactions occur. |
| 5 | Post-pinch | Column expands, instabilities develop, radiation decays. |

In the Lee model, the transition from phase 2 → 3 happens when `z_sheet` reaches the anode tip (`z = anode_length`). The transition from phase 3 → 4 happens when the shock radius reaches its minimum (`vr = 0`).

In [ ]:
t_us = result.t * 1e6
I_MA = result.I / 1e6
z_cm = result.z_sheet * 100
r_cm = result.r_shock * 100

# Identify phase boundaries from the position arrays
# Phase 2 ends when z_sheet first hits anode_length
anode_length_m = sp['anode_length']
rundown_end_idx = np.argmax(z_cm >= anode_length_m * 100 * 0.99)
if rundown_end_idx == 0:
    rundown_end_idx = len(t_us) // 3   # fallback
t_rundown_end = t_us[rundown_end_idx]

# Phase 3 ends at pinch time
t_pinch_us = result.pinch_time * 1e6

# Create a composite figure
fig = plt.figure(figsize=(14, 8))
fig.suptitle('PF-1000 — Five Discharge Phases', fontsize=14, fontweight='bold')

gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.35)
ax_I = fig.add_subplot(gs[0, :])
ax_z = fig.add_subplot(gs[1, 0])
ax_r = fig.add_subplot(gs[1, 1])

# --- Current trace with phase shading ---
ax_I.plot(t_us, I_MA, 'k', linewidth=2)
ax_I.axvline(0.0,            color='C4', linestyle='--', linewidth=1)
ax_I.axvline(t_rundown_end,  color='C2', linestyle='--', linewidth=1)
ax_I.axvline(t_pinch_us,     color='C3', linestyle='--', linewidth=1)

# Shade phases
t_end = t_us[-1]
ax_I.axvspan(0,            t_rundown_end, alpha=0.07, color='C2', label='Phase 2 — rundown')
ax_I.axvspan(t_rundown_end, t_pinch_us,  alpha=0.07, color='C3', label='Phase 3 — radial')
ax_I.axvspan(t_pinch_us,    t_end,        alpha=0.07, color='C1', label='Phase 4+5 — pinch/post')

# Annotations
mid_rundown = t_rundown_end / 2
ax_I.text(mid_rundown, I_MA.max() * 0.30, 'Phase 2\nRundown', ha='center', fontsize=8, color='C2')
mid_radial = (t_rundown_end + t_pinch_us) / 2
ax_I.text(mid_radial, I_MA.max() * 0.50, 'Phase 3\nRadial', ha='center', fontsize=8, color='C3')
ax_I.text(t_pinch_us * 1.03, I_MA.max() * 0.30, 'Phase 4\nPinch', ha='left', fontsize=8, color='C1')

ax_I.set_ylabel('Current (MA)', fontsize=11)
ax_I.set_xlabel('Time (µs)', fontsize=11)
ax_I.legend(loc='upper right', fontsize=9)
ax_I.grid(True, alpha=0.3)

# --- Axial position ---
ax_z.plot(t_us[:rundown_end_idx+1], z_cm[:rundown_end_idx+1], 'C2', linewidth=2)
ax_z.axhline(anode_length_m * 100, color='gray', linestyle='--', linewidth=1, label='Anode tip')
ax_z.set_xlabel('Time (µs)', fontsize=10)
ax_z.set_ylabel('Sheet position z (cm)', fontsize=10)
ax_z.set_title('Phase 2: Axial rundown')
ax_z.legend(fontsize=9)
ax_z.grid(True, alpha=0.3)

# --- Radial position ---
ax_r.plot(t_us[rundown_end_idx:], r_cm[rundown_end_idx:], 'C3', linewidth=2)
ax_r.axhline(cc['cathode_radius'] * 100, color='gray', linestyle='--', linewidth=1, label='Cathode')
ax_r.axhline(cc['anode_radius'] * 100, color='sienna', linestyle=':', linewidth=1, label='Anode')
ax_r.set_xlabel('Time (µs)', fontsize=10)
ax_r.set_ylabel('Shock radius r (cm)', fontsize=10)
ax_r.set_title('Phase 3: Radial implosion')
ax_r.legend(fontsize=9)
ax_r.grid(True, alpha=0.3)

plt.show()

print(f"Phase boundary times:")
print(f"  Phase 2 starts : ~0 µs (liftoff)")
print(f"  Phase 3 starts : {t_rundown_end:.2f} µs  (sheet reaches anode tip)")
print(f"  Phase 4 starts : {t_pinch_us:.2f} µs  (pinch minimum radius)")

## Step 4: What do fc and fm mean physically?

These two numbers are the Lee model's two free parameters:

**fc — current fraction**  
In a real DPF shot, not all the current flows in the main current sheet. Some current arcs diffusely through the bulk gas between the electrodes, and some flows as return current along the electrode surfaces. The effective magnetic force on the sheet is:

> F_mag = (µ₀/4π) · ln(b/a) · (fc · I)²

fc accounts for the fraction that actually drives the sheet. Typical value: **0.6–0.8**. A low fc means much of the current "leaks" past the sheet.

**fm — mass fraction**  
The current sheet is not a perfect snowplow. It does not collect 100% of the gas it passes through. Some gas slips around the edges or through the sheet due to Rayleigh-Taylor instabilities. fm is the fraction actually swept:

> m_swept(z) = fm · ρ₀ · π · (b² - a²) · z

Typical value: **0.05–0.25**. A high fm means the sheet is thick and sweeps efficiently.

In [ ]:
# Show how fc and fm change the waveform shape for PF-1000
fc_cases = [0.60, 0.70, 0.80]
fm_fixed = sp['mass_fraction']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Effect of fc and fm on PF-1000 I(t)', fontsize=13)

# Vary fc
for fc_val, color in zip(fc_cases, ['C2', 'C0', 'C3']):
    m = LeeModel(current_fraction=fc_val, mass_fraction=fm_fixed)
    r = m.run(device_params=device_params)
    axes[0].plot(r.t * 1e6, r.I / 1e6, color=color, linewidth=2,
                 label=f'fc={fc_val:.2f}  I_peak={r.peak_current/1e6:.2f} MA')

axes[0].set_xlabel('Time (µs)', fontsize=11)
axes[0].set_ylabel('Current (MA)', fontsize=11)
axes[0].set_title('Varying fc (fm fixed)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Vary fm
fc_fixed = sp['current_fraction']
fm_cases = [0.05, 0.08, 0.15]

for fm_val, color in zip(fm_cases, ['C2', 'C0', 'C3']):
    m = LeeModel(current_fraction=fc_fixed, mass_fraction=fm_val)
    r = m.run(device_params=device_params)
    axes[1].plot(r.t * 1e6, r.I / 1e6, color=color, linewidth=2,
                 label=f'fm={fm_val:.2f}  t_pinch={r.pinch_time*1e6:.1f} µs')

axes[1].set_xlabel('Time (µs)', fontsize=11)
axes[1].set_ylabel('Current (MA)', fontsize=11)
axes[1].set_title('Varying fm (fc fixed)')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observations:")
print("  fc larger → higher I_peak (more current drives the sheet, faster rundown, more current at pinch)")
print("  fm larger → slower rundown (heavier slug), pinch occurs later in time")
print("  Both shift the timing and depth of the current dip")

## Step 5: Compare against published PF-1000 experimental data

The simulator database includes a digitized I(t) waveform from published PF-1000 shots. We can overlay the simulation directly onto the experimental data to see how well the Lee model reproduces the real discharge.

In [ ]:
exp = DEVICES['PF-1000']

fig, ax = plt.subplots(figsize=(11, 5))

# Experimental waveform
ax.plot(exp.waveform_t * 1e6, exp.waveform_I / 1e6,
        'ko-', markersize=5, linewidth=1.5, label='PF-1000 experimental (digitized)')

# Simulation
ax.plot(result.t * 1e6, result.I / 1e6,
        'C0', linewidth=2, label='Lee model simulation')

# Mark peak
ax.axvline(result.peak_current_time * 1e6, color='C0', linestyle=':', linewidth=1)
ax.axvline(exp.current_rise_time * 1e6 if exp.current_rise_time else exp.waveform_t[np.argmax(exp.waveform_I)] * 1e6,
           color='gray', linestyle=':', linewidth=1)

ax.set_xlabel('Time (µs)', fontsize=12)
ax.set_ylabel('Current (MA)', fontsize=12)
ax.set_title('PF-1000: Lee Model vs Experiment', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Quantify the agreement
sim_peak    = result.peak_current / 1e6
exp_peak    = exp.peak_current / 1e6
peak_error  = abs(sim_peak - exp_peak) / exp_peak * 100

print(f"Peak current comparison:")
print(f"  Simulation : {sim_peak:.3f} MA")
print(f"  Experiment : {exp_peak:.3f} MA")
print(f"  Error      : {peak_error:.1f}%")
print()
print("The Lee model captures the overall waveform shape well.")
print("Residual error comes from circuit parasitics not captured in the 0D model.")
print("Notebook 03 shows how to calibrate fc and fm to minimize this error.")

## Step 6: Neutron yield — when do neutrons appear?

Neutron production in a DPF is highly localized in time. Neutrons are produced only during the **pinch phase**, when the plasma density and temperature are high enough for DD fusion reactions. The reaction is:

> D + D → ³He + n  (2.45 MeV neutron)  [50%]
> D + D → T + p                         [50%]

The fusion rate per unit volume is `n² · ⟨σv⟩`, where `n` is the deuterium number density and `⟨σv⟩` is the velocity-averaged cross-section — a steep function of temperature (kT in keV). Because the pinch lasts only ~10–50 nanoseconds, the neutron burst has a very narrow time profile. This is useful: it allows time-of-flight measurements that give the neutron energy spectrum.

In [ ]:
from dpf.validation.lee_model_comparison import estimate_neutron_yield_from_lee_result

# Estimate total yield from the pinch geometry
Yn_total = estimate_neutron_yield_from_lee_result(result)

print(f"Estimated total DD neutron yield: {Yn_total:.2e}")
print()

# Published PF-1000 typical yield is ~10^10 - 10^11 neutrons/shot
Yn_exp_low  = 1e10
Yn_exp_high = 2e11
print(f"Published PF-1000 yield range : {Yn_exp_low:.0e} – {Yn_exp_high:.0e} neutrons/shot")
print(f"Model estimate                : {Yn_total:.1e} neutrons/shot")
print()

# Yield timing: neutrons appear in a burst at the pinch
print(f"Neutron burst timing:")
print(f"  Pinch occurs at    : {result.pinch_time*1e6:.2f} µs")
print(f"  Pinch duration     : ~10-50 ns (estimated from Lee model pinch geometry)")
print(f"  Neutron burst width: ~5-20 ns FWHM (shorter than pinch duration)")
print()
print("The burst is so short (< 50 ns) that it is invisible on a microsecond oscilloscope.")
print("Dedicated neutron detectors (BF₃, Ag-activation, or scintillator+PMT) are needed.")

## Step 7: Beam-target vs thermonuclear fusion

There are two distinct fusion mechanisms active in a DPF pinch:

**Thermonuclear (TN):** Both fuel ions have a Maxwellian velocity distribution at temperature T. The reaction rate depends only on the bulk temperature. This is the mechanism that powers stars.

**Beam-target (BT):** A population of fast (non-thermal) ions — accelerated by the large electric fields during the pinch — streams into the cold dense background gas and fuses with it. The beam ions have energies of tens to hundreds of keV. The cross-section at these energies is orders of magnitude larger than TN at the same average temperature.

In DPF devices below ~1 MJ, **beam-target dominates** by a large factor — typically 90–99% of the observed yield. This is both good (higher yield than pure TN would give) and a caution for fusion energy applications (the beam ions are not in equilibrium, so the "plasma temperature" alone understates the reactivity).

The `YieldTracker` class accumulates both contributions separately during a simulation.

In [ ]:
# Run the YieldTracker with synthetic pinch conditions derived from the Lee model result
# We simulate a simple time series representing the pinch compression

# Extract pinch geometry from the Lee model metadata
meta = result.metadata
a    = meta['anode_radius']     # anode radius [m]
b    = meta['cathode_radius']   # cathode radius [m]
z_f  = meta['anode_length'] * sp.get('pinch_column_fraction', 0.14)  # column length
rho0_fill = meta['rho0']        # fill gas density [kg/m³]
I_pinch   = result.peak_current # current at pinch [A]

# Pinch parameters
r_min      = 0.1 * a   # minimum pinch radius (Lee standard)
pi_const   = np.pi
mu_0       = 4 * pi_const * 1e-7

# Estimate pinch density from cylindrical compression
m_D        = 3.344e-27  # deuterium atom mass [kg]
M_slug     = meta['fm'] * rho0_fill * pi_const * (b**2 - r_min**2) * z_f
V_pinch    = pi_const * r_min**2 * z_f
n_D_pinch  = M_slug / (m_D * V_pinch)  # number density of D ions [m⁻³]

# Estimate pinch temperature from Alfven speed
B_theta    = mu_0 * I_pinch / (2 * pi_const * r_min)
rho_pinch  = M_slug / V_pinch
v_A        = B_theta / np.sqrt(mu_0 * rho_pinch)   # Alfven speed [m/s]
T_pinch_keV = 0.5 * m_D * v_A**2 / (1.602e-16)     # in keV

print(f"Pinch parameters (estimated from Lee model geometry):")
print(f"  Pinch radius r_min   : {r_min*100:.2f} cm")
print(f"  Column length z_f    : {z_f*100:.1f} cm")
print(f"  Pinch density n_D    : {n_D_pinch:.2e} m⁻³")
print(f"  Alfven temperature   : {T_pinch_keV:.2f} keV")

# Now use YieldTracker to split thermonuclear vs beam-target
tracker = YieldTracker(ion_mass=m_D, rho0=rho0_fill)

# Simulate the pinch burst: 20 ns duration, 0.5 ns steps
dt_pinch  = 0.5e-9   # [s]
n_pinch_steps = 40   # 20 ns total

# State dict that YieldTracker expects
state = {
    'rho':      np.full((1, 1, 1), rho_pinch),        # density array
    'Te':       np.full((1, 1, 1), T_pinch_keV * 1.602e-16 / 1.381e-23),   # temp in K
    'Ti':       np.full((1, 1, 1), T_pinch_keV * 1.602e-16 / 1.381e-23),
    'pressure': np.full((1, 1, 1), 2 * n_D_pinch * 1.381e-23 * T_pinch_keV * 1e3 * 11604),
}

for step in range(n_pinch_steps):
    tracker.accumulate(state=state, dt=dt_pinch, I_current=I_pinch, cell_volume=V_pinch)

yield_result = tracker.get_result()
print()
print(tracker.summary())

In [ ]:
# Build time-resolved yield profile
t_pinch_ns  = np.arange(len(yield_result.dY_thermo)) * dt_pinch * 1e9   # ns
dY_thermo   = np.array(yield_result.dY_thermo)
dY_bt       = np.array(yield_result.dY_bt)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Neutron Yield Decomposition — PF-1000 Pinch', fontsize=13)

# Time-resolved yield rate
ax1.fill_between(t_pinch_ns, dY_bt, alpha=0.6, color='C0', label='Beam-target (BT)')
ax1.fill_between(t_pinch_ns, dY_thermo, alpha=0.6, color='C3', label='Thermonuclear (TN)')
ax1.set_xlabel('Time after pinch start (ns)', fontsize=11)
ax1.set_ylabel('Neutrons per step', fontsize=11)
ax1.set_title('Neutron production rate')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Cumulative yield
ax2.plot(t_pinch_ns, np.array(yield_result.Y_bt_cumulative),  'C0', linewidth=2, label='BT cumulative')
ax2.plot(t_pinch_ns, np.array(yield_result.Y_thermo_cumulative), 'C3', linewidth=2, label='TN cumulative')
ax2.set_xlabel('Time after pinch start (ns)', fontsize=11)
ax2.set_ylabel('Cumulative neutrons', fontsize=11)
ax2.set_title('Cumulative yield')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Yn_thermo = yield_result.Y_thermo_cumulative[-1] if yield_result.Y_thermo_cumulative else 0
Yn_bt     = yield_result.Y_bt_cumulative[-1]     if yield_result.Y_bt_cumulative else 0
Yn_total  = Yn_thermo + Yn_bt

if Yn_total > 0:
    print(f"Yield breakdown over 20 ns pinch:")
    print(f"  Thermonuclear : {Yn_thermo:.2e}  ({Yn_thermo/Yn_total*100:.1f}%)")
    print(f"  Beam-target   : {Yn_bt:.2e}  ({Yn_bt/Yn_total*100:.1f}%)")
    print(f"  Total         : {Yn_total:.2e}")
    print()
    print("Conclusion: In kJ–MJ DPF devices, beam-target dominates.")
    print("At very high temperatures (T > 10 keV) or higher stored energy, TN fraction grows.")

## Step 8: How does yield scale with peak current?

Lee & Saw (2008) showed empirically from many DPF devices that:

> Yn ∝ I_pinch⁴   (for kJ–MJ class devices, deuterium fill)

This steep scaling means doubling the current increases yield by 16×. It is why large DPFs (PF-1000, MJOLNIR) are built: more stored energy → higher current → disproportionately higher yield.

In [ ]:
# Scan voltage (and therefore approximate current) to show I⁴ scaling
voltages_kV  = np.array([10, 15, 20, 25, 30])   # kV
I_peaks_kA   = []
Yn_estimates = []

for V0_kV in voltages_kV:
    params_v = dict(device_params)
    params_v['V0'] = V0_kV * 1e3
    m = LeeModel(current_fraction=sp['current_fraction'], mass_fraction=sp['mass_fraction'])
    r = m.run(device_params=params_v)
    yn = estimate_neutron_yield_from_lee_result(r)
    I_peaks_kA.append(r.peak_current / 1e3)
    Yn_estimates.append(yn)

I_peaks_kA   = np.array(I_peaks_kA)
Yn_estimates = np.array(Yn_estimates)

# Fit a power law Yn = A * I^n
log_I = np.log(I_peaks_kA)
log_Y = np.log(Yn_estimates)
n_fit, log_A = np.polyfit(log_I, log_Y, 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.loglog(I_peaks_kA, Yn_estimates, 'C0o', markersize=9, label='Lee model estimates')

# Overlay power-law fit
I_fit = np.linspace(I_peaks_kA.min() * 0.9, I_peaks_kA.max() * 1.1, 50)
Y_fit = np.exp(log_A) * I_fit**n_fit
ax.loglog(I_fit, Y_fit, 'C3--', linewidth=1.5, label=f'Power-law fit: Yn ∝ I^{n_fit:.2f}')

for V, I, Y in zip(voltages_kV, I_peaks_kA, Yn_estimates):
    ax.annotate(f'{V} kV', (I, Y), textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_xlabel('Peak current I_peak (kA)', fontsize=11)
ax.set_ylabel('Estimated neutron yield Yn', fontsize=11)
ax.set_title('Neutron yield scaling with current (PF-1000 geometry)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Fitted power law exponent: Yn ∝ I^{n_fit:.2f}")
print(f"Lee & Saw (2008) empirical: Yn ∝ I^4 (for kJ-MJ devices)")
print(f"Agreement is qualitative — exact exponent depends on fill pressure and geometry.")

## Summary

In this notebook you learned:

1. **Five phases** of a DPF discharge: breakdown → rundown → radial → pinch → post-pinch
2. **fc and fm** are the Lee model's physical parameters: current fraction and mass fraction
3. **Quantitative comparison** against PF-1000 experimental data: ~5–18% peak current error from a 0D model is typical
4. **Neutron yield** is highly localized to the pinch (< 50 ns)
5. **Beam-target fusion dominates** in kJ–MJ devices; thermonuclear fraction grows only at very high energy
6. **Yield scales as I⁴** — peak current is the most important design variable

---

**Next:** Notebook 03 shows how to systematically sweep parameters and auto-calibrate fc/fm against experimental data.